# Tests des fonctions de calcul de la saturation - full chunk

découpage 1 : à partir des pools
découpage 2 : à partir des stations (après élimination des stations déja traitées)

In [8]:
import sys
from datetime import date, datetime
from pathlib import Path

import geopandas as gpd
import pandas as pd
from e1 import e1

sys.path.insert(0, str(Path.cwd().parent))
from constants import TILES_IGN, ATTR_IGN

CHUNK_SIZE: int = 20 #0
SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75

#day = date(2026,7, 5)
#day = date(2026,7, 14)
day = date(2026,8, 1)
#day = date(2026,8, 18)
#day = date(2026,8, 25)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

## test qualicharge

In [2]:
def surfaces_echangeurs(
    points_stations: gpd.GeoDataFrame,
    echangeurs: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """Return the convex hull surfaces of stations and echangeurs for each pool."""

    points = pd.concat([points_stations[[ID_POOL, "geometry"]], echangeurs[[ID_POOL, "geometry"]]], ignore_index=True)

    surfaces = points.to_crs(epsg=2154).dissolve(by=ID_POOL)
    surfaces["geometry"] = surfaces.geometry.convex_hull.buffer(BUFFER_ECHANGEUR)
  
    return surfaces.to_crs(epsg=4326).reset_index()

In [3]:
day = datetime(2026,8, 1)
indicators = e1(day)

e5  4310979
67596


In [4]:
indicators

,target,value,code,period,timestamp,category,extras
0,00,3568,e1,d,2026-08-01,None,{'id_station_itinerance': ['FRHPCPNF080371TIER...


In [7]:
ech_station = pd.DataFrame(indicators.loc[0, 'extras']).sort_values(by="id_pool")
echangeurs = ech_station[ech_station["id_pool"].str[0:1] == "E"]
aires = ech_station[ech_station["id_pool"].str[0:1] == "A"]
len(echangeurs), len(aires)

(3026, 542)